In [ ]:
import matplotlib.pyplot as plt
import minisom
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST, FashionMNIST
from torchvision.transforms import v2

from models import LeNet5, FberNet

np.random.seed(148726465)
torch.manual_seed(9921960169883554127);

In [ ]:
transform = v2.Compose([v2.ToImage(), v2.ToDtype(dtype=torch.float32), v2.Normalize(mean=[0], std=[1])])

test_dataset = MNIST(root='data/', train=False, transform=transform, download=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
state_dict = torch.load('fbernet_mnist.pth', weights_only=True)
model = FberNet()
model.load_state_dict(state_dict)
model.eval();

In [ ]:
train_conv5_data = np.empty((len(test_dataset), 30))
train_sub2_data = np.empty((len(test_dataset), 3, 196))
train_sub4_data = np.empty((len(test_dataset), 10, 25))
train_targets = np.empty(len(test_dataset))
train_imgs = np.empty((len(test_dataset), 28, 28))

with torch.no_grad():
    for i, (images, targets) in enumerate(test_loader):
        start = 64 * i
        end = start + 64
        x = images
        x = model.conv1(x)
        x = model.sub2(x)
        train_sub2_data[start:end, :] = x.numpy().squeeze().reshape(-1, 3, 196)
        x = model.conv3(x)
        x = model.sub4(x)
        train_sub4_data[start:end, :] = x.numpy().squeeze().reshape(-1, 10, 25)
        x = model.conv5(x)
        train_conv5_data[start:end, :] = x.numpy().squeeze()
        train_targets[start:end] = targets.numpy().squeeze()
        train_imgs[start:end] = test_dataset.transform(images).numpy().reshape(-1, 28, 28)

In [ ]:
grid_size = 80
som = minisom.MiniSom(grid_size, grid_size, 30, sigma=10.0, learning_rate=1.0, random_seed=2069184213)
som.train(train_conv5_data, num_iteration=10000, verbose=True)

In [ ]:
wmap = {}
im = 0
for x, t in zip(train_conv5_data, train_targets):
    w = som.winner(x)
    wmap[w] = im
    im += 1

In [ ]:
plt.figure(figsize=(12, 12))
for x, t in zip(train_conv5_data, train_targets):
    w = som.winner(x)
    plt.text(w[0]+.5,  w[1]+.5,  str(int(t)),
             color=plt.cm.rainbow(t / 10.), fontdict={'size': 9})
plt.axis([0, som.get_weights().shape[0], 0,  som.get_weights().shape[1]])
plt.show()

In [ ]:
image = np.zeros((grid_size * 28, grid_size * 28))

for j in range(grid_size):
    for i in range(grid_size):
        if (i, j) in wmap:
            image[28 * j:28 * (j + 1), 28 * i:28 * (i + 1)] = train_imgs[wmap[(i, j)], ::-1, :]

plt.figure(figsize=(30, 30))
plt.imshow(image, cmap='Greys', interpolation='nearest', origin='lower')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
grid_size = 80
som = minisom.MiniSom(grid_size, grid_size, 25, sigma=5.0, learning_rate=0.5, random_seed=2069184213)
data = train_sub4_data[:, 0, :]
data = (data - np.mean(data)) / np.std(data)
som.train(data, num_iteration=10000, verbose=True)

In [ ]:
wmap = {}
im = 0
for x, t in zip(data, train_targets):
    w = som.winner(x)
    wmap[w] = im
    im += 1

In [ ]:
plt.figure(figsize=(12, 12))
for x, t in zip(data, train_targets):
    w = som.winner(x)
    plt. text(w[0]+.5,  w[1]+.5,  str(int(t)),
              color=plt.cm.rainbow(t / 10.), fontdict={'size': 9})
plt.axis([0, som.get_weights().shape[0], 0,  som.get_weights().shape[1]])
plt.show()

In [ ]:
image = np.zeros((grid_size * 28, grid_size * 28))

for j in range(grid_size):
    for i in range(grid_size):
        if (i, j)in wmap:
            image[28 * j:28 * (j + 1), 28 * i:28 * (i + 1)] = train_imgs[wmap[(i, j)], ::-1, :]

plt.figure(figsize=(30, 30))
plt.imshow(image, cmap='Greys', interpolation='nearest', origin='lower')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
grid_size = 80
som = minisom.MiniSom(grid_size, grid_size, 196, sigma=5.0, learning_rate=0.5, random_seed=2069184213)
data = train_sub2_data[:, 1, :]
data = (data - np.mean(data)) / np.std(data)
som.train(data, num_iteration=10000, verbose=True)

In [ ]:
wmap = {}
im = 0
for x, t in zip(data, train_targets):
    w = som.winner(x)
    wmap[w] = im
    im += 1

In [ ]:
plt.figure(figsize=(12, 12))
for x, t in zip(data, train_targets):
    w = som.winner(x)
    plt.text(w[0]+.5,  w[1]+.5,  str(int(t)),
              color=plt.cm.rainbow(t / 10.), fontdict={'size': 9})
plt.axis([0, som.get_weights().shape[0], 0,  som.get_weights().shape[1]])
plt.show()

In [ ]:
image = np.zeros((grid_size * 28, grid_size * 28))

for j in range(grid_size):
    for i in range(grid_size):
        if (i, j)in wmap:
            image[28 * j:28 * (j + 1), 28 * i:28 * (i + 1)] = train_imgs[wmap[(i, j)], ::-1, :]

plt.figure(figsize=(30, 30))
plt.imshow(image, cmap='Greys', interpolation='nearest', origin='lower')
plt.axis('off')
plt.tight_layout()
plt.show()